In [1]:
!pip install -q open_clip_torch
!pip install -q hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numb

In [2]:
import open_clip
import torch

print(torch.cuda.is_available())

True


In [3]:
import torch
import open_clip
import numpy as np

from pathlib import Path
from PIL import Image
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

cuda


In [4]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14",
    pretrained="openai"
)

model = model.to(device)
model.eval()

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-23): 24 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1024,), eps=1e-05, elementwi

In [5]:
from pathlib import Path

image_dir = Path(
    "/kaggle/input/datasets/robinchetry/cluster0-images"
)

image_paths = sorted(
    list(image_dir.rglob("*"))
)

image_paths = [
    p for p in image_paths
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

print("Images:", len(image_paths))

Images: 26397


In [6]:
image_dir = Path(
    "/kaggle/input/datasets/robinchetry/cluster0-images"
)

image_paths = sorted(
    list(image_dir.glob("*"))
)

print(len(image_paths))

26397


In [ ]:
features = []

valid_paths = []

with torch.no_grad():

    for path in tqdm(image_paths):

        try:

            img = Image.open(path).convert("RGB")

            img = preprocess(img).unsqueeze(0).to(device)

            feat = model.encode_image(img)

            feat = feat / feat.norm(
                dim=-1,
                keepdim=True
            )

            features.append(
                feat.cpu().numpy()[0]
            )

            valid_paths.append(
                str(path)
            )

        except:

            continue


features = np.array(features)

print(features.shape)

In [ ]:
np.save(
    "clip_features.npy",
    features
)

np.save(
    "clip_paths.npy",
    valid_paths
)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=250,
    random_state=42
)

pca_features = pca.fit_transform(
    features
)

print(
    pca.explained_variance_ratio_.sum()
)

print(
    pca_features.shape
)

In [ ]:
import umap

umap_model = umap.UMAP(

    n_neighbors=100,

    min_dist=0.0,

    n_components=20,

    metric="cosine",

    random_state=42
)

umap_features = umap_model.fit_transform(
    pca_features
)

print(
    umap_features.shape
)

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(

    min_cluster_size=300,

    min_samples=100,

    metric="euclidean",

    cluster_selection_method="eom"
)

labels = clusterer.fit_predict(
    umap_features
)

print(
    np.unique(labels)
)

In [ ]:
    import pandas as pd

stats = pd.Series(labels)

print(
    stats.value_counts()
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

plt.scatter(

    umap_features[:,0],

    umap_features[:,1],

    c=labels,

    cmap="tab20",

    s=2
)

plt.title(
    "CLIP Cluster0 Reclustering"
)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import random

df = pd.DataFrame({
    "image_path": image_paths,
    "cluster": labels
})

In [ ]:
def show_cluster_random(cluster_id, n=20):

    paths = df[
        df["cluster"] == cluster_id
    ]["image_path"].tolist()

    if len(paths) == 0:
        print(f"Cluster {cluster_id} empty")
        return

    paths = random.sample(
        paths,
        min(n, len(paths))
    )

    cols = 5
    rows = (len(paths) + cols - 1) // cols

    plt.figure(
        figsize=(15, rows * 3)
    )

    for i, path in enumerate(paths):

        img = Image.open(path)

        plt.subplot(
            rows,
            cols,
            i + 1
        )

        plt.imshow(img)
        plt.axis("off")

    plt.suptitle(
        f"Cluster {cluster_id}",
        fontsize=18
    )

    plt.tight_layout()
    plt.show()

In [ ]:
for c in sorted(df.cluster.unique()):

    print(
        f"\nCluster {c}"
    )

    show_cluster_random(
        c,
        n=20
    )

# **experimentation with ocr and clip combination on 1000 images**

In [ ]:
!pip install -q easyocr


In [ ]:
import easyocr

reader = easyocr.Reader(
    ['en'],
    gpu=True
)

In [ ]:
print(type(image_paths[0]))
print(image_paths[0])


In [ ]:
import cv2

img = cv2.imread(
    str(image_paths[0])
)

result = reader.readtext(
    img,
    detail=0
)
result=clean_text(result)

print(result)

In [ ]:
import random

for idx in random.sample(
    range(len(image_paths)),
    10
):

    txt = reader.readtext(
        str(image_paths[idx]),
        detail=0
    )

    print("\n================")
    print("IMAGE:", idx)
    print(txt[:20])

In [ ]:
import re

def clean_text(words):

    filtered = []

    for w in words:

        w = w.strip()

        # Remove very short tokens
        if len(w) < 3:
            continue

        # Remove pure numbers
        if re.fullmatch(r"[0-9.\-]+", w):
            continue

        filtered.append(w)

    return " ".join(filtered)

# Let's do a small scientific experiment first, not the full 26k images.

In [ ]:
import random

sample_paths = random.sample(
    image_paths,
    100
)

len(sample_paths)

In [ ]:
image_features = []

with torch.no_grad():

    for path in sample_paths:

        img = Image.open(
            str(path)
        ).convert("RGB")

        img = preprocess(img)\
            .unsqueeze(0)\
            .to(device)

        feat = model.encode_image(img)

        feat = feat / feat.norm(
            dim=-1,
            keepdim=True
        )

        image_features.append(
            feat.cpu().numpy()[0]
        )

image_features = np.array(
    image_features
)

print(image_features.shape)

In [ ]:
texts = []

for path in sample_paths:

    try:

        words = reader.readtext(
            str(path),
            detail=0
        )

        txt = " ".join(words)

    except:

        txt = ""

    texts.append(txt)

print(texts[:3])

In [ ]:
text_features = []

with torch.no_grad():

    for txt in texts:

        if txt.strip() == "":

            text_features.append(
                np.zeros(768)
            )

            continue

        tokens = open_clip.tokenize(
            [txt]
        ).to(device)

        feat = model.encode_text(
            tokens
        )

        feat = feat / feat.norm(
            dim=-1,
            keepdim=True
        )

        text_features.append(
            feat.cpu().numpy()[0]
        )

text_features = np.array(
    text_features
)

print(text_features.shape)

In [ ]:
combined_features = np.concatenate(

    [
        image_features,
        0.3 * text_features
    ],

    axis=1
)

print(combined_features.shape)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=50,
    random_state=42
)

pca_features = pca.fit_transform(
    combined_features
)

print(pca_features.shape)

In [ ]:
import umap

umap_model = umap.UMAP(

    n_neighbors=15,

    min_dist=0.0,

    metric="cosine",

    random_state=42
)

umap_features = umap_model.fit_transform(
    pca_features
)

print(umap_features.shape)

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(

    min_cluster_size=5,

    min_samples=3
)

labels = clusterer.fit_predict(
    umap_features
)

print(
    np.unique(labels)
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

plt.scatter(

    umap_features[:,0],

    umap_features[:,1],

    c=labels,

    cmap="tab20",

    s=40
)

plt.title(
    "CLIP + OCR Clustering"
)

plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

def show_cluster_random(df, cluster_id, n=20):

    paths = df[
        df["cluster"] == cluster_id
    ]["image_path"].tolist()

    if len(paths) == 0:
        print("Empty cluster")
        return

    paths = random.sample(
        paths,
        min(n, len(paths))
    )

    cols = 5
    rows = (len(paths) + cols - 1) // cols

    plt.figure(
        figsize=(15, rows * 3)
    )

    for i, path in enumerate(paths):

        img = Image.open(str(path))

        plt.subplot(
            rows,
            cols,
            i + 1
        )

        plt.imshow(img)
        plt.axis("off")

    plt.suptitle(
        f"Cluster {cluster_id}",
        fontsize=18
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

df_test = pd.DataFrame({

    "image_path": sample_paths,

    "cluster": labels

})

In [ ]:
show_cluster_random(
    df_test,
    cluster_id=-1,
    n=20
)

In [ ]:
show_cluster_random(
    df_test,
    cluster_id=0,
    n=20
)

In [ ]:
show_cluster_random(
    df_test,
    cluster_id=1,
    n=20
)

In [ ]:
show_cluster_random(
    df_test,
    cluster_id=2,
    n=20
)

In [ ]:
show_cluster_random(
    df_test,
    cluster_id=3,
    n=20
)

In [ ]:
show_cluster_random(
    df_test,
    cluster_id=4,
    n=20
)